In [ ]:
# All installs in one cell — run once per session
!pip install -q \
    requests \
    beautifulsoup4 \
    langdetect \
    spacy \
    sentence-transformers \
    chromadb \
    langchain \
    langchain-community \
    tqdm \
    python-dotenv \
    transformers \
    accelerate

!python -m spacy download en_core_web_sm -q

print("All packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 22.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━

# Mount Google Drive

In [6]:
# ChromaDB data survives session resets when stored on Drive
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Create project folder on Drive
PROJECT_DIR = Path("/content/drive/MyDrive/nyaya_agent")
CHROMA_DIR  = PROJECT_DIR / "chroma_db"
RAW_DIR     = PROJECT_DIR / "data" / "raw_kanoon"

for d in [PROJECT_DIR, CHROMA_DIR, RAW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project dir: {PROJECT_DIR}")
print(f"ChromaDB dir: {CHROMA_DIR}")
print(f"Raw data dir: {RAW_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/nyaya_agent
ChromaDB dir: /content/drive/MyDrive/nyaya_agent/chroma_db
Raw data dir: /content/drive/MyDrive/nyaya_agent/data/raw_kanoon


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# API keys

In [7]:
# Use Colab's built-in secrets manager
# Left sidebar → key icon → add:
#   KANOON_API_KEY   — Indian Kanoon API
#   GROQ_API_KEY     — Groq (free tier, fast Llama-3)  https://console.groq.com/keys

from google.colab import userdata

try:
    KANOON_KEY = userdata.get('KANOON_API_KEY')
    GROQ_KEY   = userdata.get('GROQ_API_KEY')
    print("API keys loaded from Colab Secrets")
except Exception:
    import getpass
    KANOON_KEY = getpass.getpass("Indian Kanoon API key: ")
    GROQ_KEY   = getpass.getpass("Groq API key: ")
    print("API keys entered manually")


API keys loaded from Colab Secrets


# Fetcher

In [8]:
import requests, time, json

KANOON_API = "https://api.indiankanoon.org"
HEADERS    = {"Authorization": f"Token {KANOON_KEY}"}

def search_judgments(query: str, max_pages: int = 5) -> list[dict]:
    docs = []
    for page in range(max_pages):
        resp = requests.post(
            f"{KANOON_API}/search/",
            data={"formInput": query, "pagenum": page},
            headers=HEADERS, timeout=30,
        )
        resp.raise_for_status()
        batch = resp.json().get("docs", [])
        if not batch:
            break
        docs.extend(batch)
        time.sleep(0.6)
    return docs

def fetch_document(tid: int) -> dict | None:
    cache = RAW_DIR / f"{tid}.json"    # Drive-backed cache
    if cache.exists():
        return json.loads(cache.read_text())
    resp = requests.post(
        f"{KANOON_API}/doc/{tid}/",
        headers=HEADERS, timeout=30,
    )
    if resp.status_code != 200:
        return None
    data = resp.json()
    cache.write_text(json.dumps(data, ensure_ascii=False))
    time.sleep(0.5)
    return data

def fetch_corpus(queries: list[str], max_per_query: int = 50) -> list[dict]:
    all_docs, seen = [], set()
    for query in queries:
        stubs = search_judgments(query, max_pages=max_per_query // 10)
        for stub in stubs:
            tid = stub.get("tid")
            if not tid or tid in seen:
                continue
            doc = fetch_document(tid)
            if doc:
                all_docs.append(doc)
                seen.add(tid)
        print(f"  '{query[:40]}': {len(seen)} docs fetched")
    return all_docs

# Fetch — start small to test, then increase max_per_query
QUERIES = [
    "insider trading SEBI penalty Supreme Court",
    "contract breach specific performance",
    "fundamental rights constitutional law",
]

print("Fetching from Indian Kanoon...")
raw_docs = fetch_corpus(QUERIES, max_per_query=50)
print(f"\nTotal: {len(raw_docs)} documents")

Fetching from Indian Kanoon...
  'insider trading SEBI penalty Supreme Cou': 50 docs fetched
  'contract breach specific performance': 100 docs fetched
  'fundamental rights constitutional law': 150 docs fetched

Total: 150 documents


# Preprocessing + chunking

In [9]:
import re, unicodedata, spacy
from bs4 import BeautifulSoup
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 42

nlp = spacy.load("en_core_web_sm")
nlp.max_length = 2_000_000

BOILERPLATE_RE = re.compile(
    r'Take notes as you read.*?judgment|Print this judgment|'
    r'Download this judgment|Try out our Premium.*?FREE',
    re.IGNORECASE | re.DOTALL
)
CITATION_NORMALISE = [
    (re.compile(r'\(\s*(\d{4})\s*\)'),     r'(\1)'),
    (re.compile(r'S\s*\.?\s*C\s*\.?\s*C'), r'SCC'),
    (re.compile(r'A\s*\.?\s*I\s*\.?\s*R'), r'AIR'),
]
HEADING_RE = re.compile(
    r'\n(?=(?:JUDGMENT|ORDER|HELD|FACTS|RATIO|OBITER|PER [A-Z]+|\d+\.\s+[A-Z]))',
    re.MULTILINE
)
LABELS_MAP = {
    "JUDGMENT":"judgment","ORDER":"order","HELD":"ratio",
    "FACTS":"facts","RATIO":"ratio","OBITER":"obiter","PER ":"opinion"
}

def extract_and_preprocess(raw: dict) -> dict | None:
    html  = raw.get("doc", "")
    soup  = BeautifulSoup(html, "html.parser")
    for tag in soup(["script","style","nav","button"]): tag.decompose()
    text  = soup.get_text(separator="\n")
    text  = BOILERPLATE_RE.sub("", text)
    text  = re.sub(r'\n{3,}', '\n\n', text)
    text  = re.sub(r'[ \t]{2,}', ' ', text)
    text  = re.sub(r'\u00a0', ' ', text)
    text  = unicodedata.normalize("NFKC", text).strip()
    for pat, rep in CITATION_NORMALISE: text = pat.sub(rep, text)
    if len(text) < 500: return None
    try:
        lang = detect(text[:2000])
    except Exception:
        lang = "en"
    ascii_ratio = sum(1 for c in text if ord(c) < 128) / max(len(text), 1)
    if lang not in ("en","unknown") and ascii_ratio < 0.60: return None
    return {
        "tid":   str(raw.get("tid","")),
        "title": raw.get("title","")[:300],
        "court": raw.get("court",""),
        "year":  (raw.get("publishdate","") or "")[:4],
        "text":  text, "language": lang,
    }

def chunk_document(doc: dict, max_tok=400, overlap_tok=64) -> list[dict]:
    parts = HEADING_RE.split(doc["text"])
    chunks, idx = [], 0
    for part in parts:
        label = "general"
        for kw, tag in LABELS_MAP.items():
            if part.strip().upper().startswith(kw):
                label = tag; break
        spacy_doc  = nlp(part[:500_000])
        sentences  = [s.text.strip() for s in spacy_doc.sents if s.text.strip()]
        window, wtok = [], 0
        for sent in sentences:
            stok = int(len(sent.split()) * 1.3)
            if wtok + stok > max_tok and window:
                chunks.append({**{k:v for k,v in doc.items() if k!="text"},
                    "chunk_text": " ".join(window),
                    "chunk_index": idx, "section": label})
                idx += 1
                overlap, otok = [], 0
                for s in reversed(window):
                    st = int(len(s.split())*1.3)
                    if otok+st > overlap_tok: break
                    overlap.insert(0, s); otok += st
                window, wtok = overlap, otok
            window.append(sent); wtok += stok
        if window:
            chunks.append({**{k:v for k,v in doc.items() if k!="text"},
                "chunk_text": " ".join(window),
                "chunk_index": idx, "section": label})
            idx += 1
    return chunks

from tqdm.notebook import tqdm   # Colab shows progress bars inline

print("Preprocessing...")
processed = []
for raw in tqdm(raw_docs, desc="preprocess"):
    doc = extract_and_preprocess(raw)
    if doc: processed.append(doc)
print(f"{len(processed)} docs after filtering")

print("\nChunking...")
all_chunks = []
for doc in tqdm(processed, desc="chunking"):
    all_chunks.extend(chunk_document(doc))
print(f"{len(all_chunks)} chunks created")

Preprocessing...


preprocess:   0%|          | 0/150 [00:00<?, ?it/s]

150 docs after filtering

Chunking...


chunking:   0%|          | 0/150 [00:00<?, ?it/s]

21928 chunks created


# Embedding

In [10]:
from sentence_transformers import SentenceTransformer
import torch, numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding on: {device}")
if device == "cpu":
    print("WARNING: CPU embedding is slow. ~40 chunks/min.")
    print("Tip: Runtime → Change runtime type → T4 GPU")

print("Loading InLegalBERT (downloads ~440 MB on first run)...")
model = SentenceTransformer("law-ai/InLegalBERT", device=device)
print("Model loaded.")

BATCH = 64 if device == "cuda" else 16

texts = [c["chunk_text"] for c in all_chunks]
print(f"\nEmbedding {len(texts)} chunks (batch={BATCH})...")

vectors = model.encode(
    texts,
    batch_size=BATCH,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2 normalise for cosine
    convert_to_numpy=True,
)
print(f"Shape: {vectors.shape}")    # (N, 768)

# Attach embeddings to chunks
for chunk, vec in zip(all_chunks, vectors):
    chunk["embedding"] = vec.tolist()

print("Embeddings attached to chunks.")

Embedding on: cuda
Loading InLegalBERT (downloads ~440 MB on first run)...


config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/534M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded.

Embedding 21928 chunks (batch=64)...


Batches:   0%|          | 0/343 [00:00<?, ?it/s]

Shape: (21928, 768)
Embeddings attached to chunks.


# ChromaDB storage

In [11]:
import chromadb
from chromadb.config import Settings

# Point ChromaDB at Google Drive — survives session resets
client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)

collection = client.get_or_create_collection(
    name="nyaya_kanoon",
    metadata={
        "hnsw:space":           "cosine",
        "hnsw:construction_ef": 200,
        "hnsw:M":               64,
    }
)
print(f"Collection ready — {collection.count()} existing docs")

# Upsert in batches of 500
BATCH = 500
for i in tqdm(range(0, len(all_chunks), BATCH), desc="storing"):
    batch = all_chunks[i : i+BATCH]
    collection.upsert(
        ids        = [f"{c['tid']}_{c['chunk_index']}" for c in batch],
        documents  = [c["chunk_text"]  for c in batch],
        embeddings = [c["embedding"]   for c in batch],
        metadatas  = [{
            "tid":     c.get("tid",""),
            "title":   c.get("title","")[:200],
            "court":   c.get("court",""),
            "year":    c.get("year",""),
            "section": c.get("section","general"),
            "language":c.get("language","en"),
        } for c in batch],
    )

print(f"\nDone. {collection.count()} total chunks stored.")

Collection ready — 24414 existing docs


storing:   0%|          | 0/44 [00:00<?, ?it/s]


Done. 24414 total chunks stored.


# RAG query with Groq / Gemini / Ollama

In [12]:
# ── RAG query — three alternative LLM backends (choose ONE) ─────────────
#
#  Option A  Groq (recommended — free tier, ~300 tokens/s, no credit card)
#            Sign up at https://console.groq.com/keys
#            Models: llama-3.3-70b-versatile | mixtral-8x7b-32768
#
#  Option B  Google Gemini via google-generativeai SDK (free tier available)
#            Get key at https://aistudio.google.com/app/apikey
#
#  Option C  Ollama local (fully offline, no API key needed)
#            Install: https://ollama.com  then: ollama pull llama3.2
#
# Set BACKEND to 'groq', 'gemini', or 'ollama'
BACKEND = "groq"

import requests as req

SYSTEM_PROMPT = """You are Nyaya Agent, a senior Indian legal research assistant.
Answer using ONLY the retrieved context. Cite every claim as [N].
If context is insufficient, say so — never invent citations."""


def _call_groq(context: str, question: str) -> str:
    """Option A — Groq free tier (llama-3.3-70b-versatile)."""
    resp = req.post(
        "https://api.groq.com/openai/v1/chat/completions",
        json={
            "model":       "llama-3.3-70b-versatile",
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": f"Context:\n\n{context}\n\nQuestion: {question}"},
            ],
            "max_tokens":  1200,
            "temperature": 0.1,
        },
        headers={
            "Authorization": f"Bearer {GROQ_KEY}",
            "Content-Type":  "application/json",
        },
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]


def _call_gemini(context: str, question: str) -> str:
    """Option B — Google Gemini (gemini-1.5-flash, free tier)."""
    try:
        import google.generativeai as genai
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-generativeai"])
        import google.generativeai as genai

    genai.configure(api_key=GEMINI_KEY)          # set GEMINI_KEY in Colab Secrets
    gemini = genai.GenerativeModel("gemini-1.5-flash")
    prompt = f"{SYSTEM_PROMPT}\n\nContext:\n\n{context}\n\nQuestion: {question}"
    response = gemini.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            max_output_tokens=1200,
            temperature=0.1,
        )
    )
    return response.text


def _call_ollama(context: str, question: str) -> str:
    """Option C — Ollama local (llama3.2, fully offline)."""
    resp = req.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "llama3.2",
            "stream": False,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": f"Context:\n\n{context}\n\nQuestion: {question}"},
            ],
            "options": {"temperature": 0.1, "num_predict": 1200},
        },
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["message"]["content"]


_BACKENDS = {
    "groq":   _call_groq,
    "gemini": _call_gemini,
    "ollama": _call_ollama,
}


def rag_query(question: str, n_results: int = 6, year_from: str = None) -> str:
    # 1. Embed query
    q_vec = model.encode(
        [question], normalize_embeddings=True
    )[0].tolist()

    # 2. Retrieve from ChromaDB
    where = {"year": {"$gte": int(year_from)}} if year_from else None
    results = collection.query(
        query_embeddings=[q_vec],
        n_results=n_results,
        ##where=where,
        include=["documents", "metadatas", "distances"],
    )

    # 3. Build context block
    context_lines = []
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ), 1):
        sim = round(1 - dist, 3)
        context_lines.append(
            (
                f"[{i}] {meta.get('title', '?')[:80]} | "
                f"{meta.get('court', '')} | {meta.get('year', '')} "
                f"| section: {meta.get('section', '')} | sim: {sim}\n"
                f"{doc[:600]}"
            )
        )
    context = "\n\n---\n\n".join(context_lines)

    # 4. Call the selected backend
    fn = _BACKENDS.get(BACKEND)
    if fn is None:
        raise ValueError(f"Unknown BACKEND={BACKEND!r}. Choose: groq | gemini | ollama")
    return fn(context, question)


# ── Test ──────────────────────────────────────────────────────────────────
print(f"Using backend: {BACKEND}")
answer = rag_query(
    "What is the penalty for insider trading under SEBI regulations?",
    n_results=6,
    year_from="2015",
)
print(answer)


Using backend: groq
According to [N4], the penalty for insider trading under SEBI regulations includes a monetary penalty imposed under Section 15G of the SEBI Act. However, the context does not provide the specific amount or details of the penalty. [N4]


###Verify Retrieval Correctness

In [ ]:
def inspect_retrieval(question: str, n_results: int = 6, year_from: str = None):
    """
    Show exactly what was retrieved BEFORE the LLM sees it.
    Use this to verify the retrieval step independently of the LLM answer.
    """
    # Embed the question
    q_vec = model.encode([question], normalize_embeddings=True)[0].tolist()

    # Retrieve from ChromaDB
    where = {"year": {"$gte": int(year_from)}} if year_from else None
    results = collection.query(
        query_embeddings=[q_vec],
        n_results=n_results,
        #where=where,
        include=["documents", "metadatas", "distances"],
    )

    print(f"Question: {question}")
    print(f"{'='*70}")
    print(f"TOP {n_results} RETRIEVED CHUNKS (before LLM sees them)")
    print(f"{'='*70}\n")

    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ), 1):
        similarity = round(1 - dist, 4)
        relevance  = "HIGH" if similarity > 0.80 else \
                     "MEDIUM" if similarity > 0.65 else "LOW"

        print(f"[{i}] Similarity: {similarity}  Relevance: {relevance}")
        print(f"     Source : {meta.get('title','?')[:80]}")
        print(f"     Court  : {meta.get('court','?')}")
        print(f"     Year   : {meta.get('year','?')}")
        print(f"     Section: {meta.get('section','?')}")
        print(f"     Text   : {doc[:300]}...")
        print()

    return results

# Run it
results = inspect_retrieval(
    "What is the penalty for insider trading under SEBI regulations?",
    n_results=6,
    year_from="2015"
)

Question: What is the penalty for insider trading under SEBI regulations?
TOP 6 RETRIEVED CHUNKS (before LLM sees them)

[1] Similarity: 0.802  Relevance: HIGH
     Source : Rakesh Agrawal vs Securities Exchange Board Of India on 1 November, 2003
     Court  : 
     Year   : 2003
     Section: general
     Text   : 114. Under the provisions of 
Section 11(2)(g)
 SEBI is to prevent insider trading and take such measures to protect the interest of investors as insider trading is per se a wrong and is prohibited. Insider Trading:-...

[2] Similarity: 0.8014  Relevance: HIGH
     Source : Rakesh Agrawal vs Securities Exchange Board Of India on 1 November, 2003
     Court  : 
     Year   : 2003
     Section: general
     Text   : 118. It is clear from the bare reading of 
Regulation 3
 that the prohibition of insider trading by an insider is an absolute offence and that benefit or gain is not an ingredient of the offence. The SEBI (Insider Trading) Regulations, 1992...

[3] Similarity: 0.8 

### Verify Answer Faithfulness

In [ ]:
def verify_faithfulness(answer: str, retrieved_results: dict) -> dict:
    """
    Check whether the generated answer is faithful to retrieved chunks.

    Method: for each sentence in the answer, check if it has high
    similarity to at least one retrieved chunk. Low similarity =
    the model may have hallucinated that sentence.
    """
    import re

    # Split answer into sentences
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', answer)
                 if len(s.strip()) > 30]

    # Get all retrieved chunk texts
    chunks = retrieved_results["documents"][0]

    print("FAITHFULNESS VERIFICATION")
    print(f"{'='*70}")
    print(f"Checking {len(sentences)} answer sentences against "
          f"{len(chunks)} retrieved chunks\n")

    results_log = []
    hallucination_flags = []

    for i, sentence in enumerate(sentences, 1):
        # Embed the answer sentence
        sent_vec = model.encode(
            [sentence], normalize_embeddings=True
        )[0]

        # Compare against all retrieved chunks
        chunk_vecs = model.encode(
            chunks, normalize_embeddings=True
        )

        # Cosine similarity between sentence and each chunk
        similarities = [
            float(sent_vec @ chunk_vec)
            for chunk_vec in chunk_vecs
        ]

        best_score   = max(similarities)
        best_chunk_i = similarities.index(best_score)
        status       = "FAITHFUL" if best_score > 0.70 else "CHECK"

        if status == "CHECK":
            hallucination_flags.append(i)

        print(f"Sentence {i}: [{status}]  Best match score: {best_score:.4f}")
        print(f"  Sentence : {sentence[:100]}...")
        if status == "CHECK":
            print(f"  WARNING  : No retrieved chunk closely supports this claim.")
            print(f"  Best chunk: {chunks[best_chunk_i][:150]}...")
        print()

        results_log.append({
            "sentence": sentence,
            "status": status,
            "best_score": best_score,
            "best_chunk_index": best_chunk_i,
        })

    # Summary
    total    = len(sentences)
    faithful = sum(1 for r in results_log if r["status"] == "FAITHFUL")
    print(f"{'='*70}")
    print(f"SUMMARY: {faithful}/{total} sentences verified as faithful")
    if hallucination_flags:
        print(f"SENTENCES TO CHECK: {hallucination_flags}")
        print("These sentences have low similarity to any retrieved chunk.")
        print("They may be hallucinated or drawn from model training memory.")
    else:
        print("All sentences are grounded in retrieved context.")

    return results_log

# First get the answer and the retrieved results together
def rag_query_with_verification(question: str, n_results: int = 6,
                                 year_from: str = None):
    """
    Full pipeline with both retrieval inspection and faithfulness check.
    """
    print("STEP 1 — RETRIEVAL INSPECTION")
    print("-" * 70)
    retrieved = inspect_retrieval(question, n_results, year_from)

    print("\nSTEP 2 — GENERATING ANSWER")
    print("-" * 70)
    answer = rag_query(question, n_results, year_from)
    print(answer)

    print("\nSTEP 3 — FAITHFULNESS VERIFICATION")
    print("-" * 70)
    verify_faithfulness(answer, retrieved)

    return answer

# Run the full verified pipeline
rag_query_with_verification(
    "What is the penalty for insider trading under SEBI regulations?",
    n_results=6,
    year_from="2015"
)

STEP 1 — RETRIEVAL INSPECTION
----------------------------------------------------------------------
Question: What is the penalty for insider trading under SEBI regulations?
TOP 6 RETRIEVED CHUNKS (before LLM sees them)

[1] Similarity: 0.802  Relevance: HIGH
     Source : Rakesh Agrawal vs Securities Exchange Board Of India on 1 November, 2003
     Court  : 
     Year   : 2003
     Section: general
     Text   : 114. Under the provisions of 
Section 11(2)(g)
 SEBI is to prevent insider trading and take such measures to protect the interest of investors as insider trading is per se a wrong and is prohibited. Insider Trading:-...

[2] Similarity: 0.8014  Relevance: HIGH
     Source : Rakesh Agrawal vs Securities Exchange Board Of India on 1 November, 2003
     Court  : 
     Year   : 2003
     Section: general
     Text   : 118. It is clear from the bare reading of 
Regulation 3
 that the prohibition of insider trading by an insider is an absolute offence and that benefit or gain is no

'According to [N4], the penalty for insider trading under SEBI regulations is a monetary penalty imposed under Section 15G of the SEBI Act. [N]'

###Verify Citation Traceability

In [ ]:
def verify_citations(answer: str, retrieved_results: dict):
    """
    Check that every [N] citation in the answer corresponds to
    an actual retrieved chunk and that the claim matches the chunk.
    """
    import re

    chunks    = retrieved_results["documents"][0]
    metadatas = retrieved_results["metadatas"][0]
    n_chunks  = len(chunks)

    # Find all citation markers in the answer
    cited_numbers = list(set(
        int(m) for m in re.findall(r'\[(\d+)\]', answer)
    ))

    print("CITATION VERIFICATION")
    print(f"{'='*70}")
    print(f"Citations found in answer: {cited_numbers}")
    print(f"Total retrieved chunks available: {n_chunks}\n")

    all_valid = True
    for num in sorted(cited_numbers):
        if num < 1 or num > n_chunks:
            print(f"[{num}] INVALID — no chunk {num} was retrieved "
                  f"(only {n_chunks} chunks available)")
            print("      This is a hallucinated citation.\n")
            all_valid = False
        else:
            meta  = metadatas[num - 1]
            chunk = chunks[num - 1]
            print(f"[{num}] VALID")
            print(f"      Source : {meta.get('title','?')[:70]}")
            print(f"      Court  : {meta.get('court','?')}")
            print(f"      Year   : {meta.get('year','?')}")
            print(f"      Preview: {chunk[:200]}...\n")

    print(f"{'='*70}")
    if all_valid:
        print("All citations verified — every [N] points to a real chunk.")
    else:
        print("WARNING: Some citations are hallucinated.")
        print("The model invented a source that does not exist.")

# Usage
answer  = rag_query(
    "What is the penalty for insider trading under SEBI regulations?",
    n_results=6
)
results = inspect_retrieval(
    "What is the penalty for insider trading under SEBI regulations?",
    n_results=6
)
verify_citations(answer, results)

Question: What is the penalty for insider trading under SEBI regulations?
TOP 6 RETRIEVED CHUNKS (before LLM sees them)

[1] Similarity: 0.802  Relevance: HIGH
     Source : Rakesh Agrawal vs Securities Exchange Board Of India on 1 November, 2003
     Court  : 
     Year   : 2003
     Section: general
     Text   : 114. Under the provisions of 
Section 11(2)(g)
 SEBI is to prevent insider trading and take such measures to protect the interest of investors as insider trading is per se a wrong and is prohibited. Insider Trading:-...

[2] Similarity: 0.8014  Relevance: HIGH
     Source : Rakesh Agrawal vs Securities Exchange Board Of India on 1 November, 2003
     Court  : 
     Year   : 2003
     Section: general
     Text   : 118. It is clear from the bare reading of 
Regulation 3
 that the prohibition of insider trading by an insider is an absolute offence and that benefit or gain is not an ingredient of the offence. The SEBI (Insider Trading) Regulations, 1992...

[3] Similarity: 0.8 

###Ground Truth Check

In [ ]:
# Define test cases where you know the correct answer
TEST_CASES = [
    {
        "question": "What is the minimum penalty for insider trading under SEBI Act?",
        "ground_truth_keywords": [
            "ten lakh", "10,00,000", "Section 15G", "minimum penalty"
        ],
        "ground_truth_facts": [
            "Minimum penalty is Rs. 10 lakh",
            "Maximum is Rs. 25 crore or 3 times profits",
            "Section 15G of the SEBI Act"
        ]
    },
    {
        "question": "What are the grounds for specific performance of a contract?",
        "ground_truth_keywords": [
            "Specific Relief Act", "discretion", "damages inadequate"
        ],
        "ground_truth_facts": [
            "Court has discretion to grant specific performance",
            "Damages must be inadequate substitute"
        ]
    }
]

def run_ground_truth_test(test_cases: list):
    """
    Run each test case and check whether the answer contains
    the expected keywords and facts.
    """
    print("GROUND TRUTH ACCURACY TEST")
    print(f"{'='*70}\n")

    for i, case in enumerate(test_cases, 1):
        question  = case["question"]
        keywords  = case["ground_truth_keywords"]

        print(f"Test {i}: {question}")
        print("-" * 50)

        answer = rag_query(question, n_results=6)
        answer_lower = answer.lower()

        # Check keyword presence
        found    = [kw for kw in keywords if kw.lower() in answer_lower]
        missing  = [kw for kw in keywords if kw.lower() not in answer_lower]

        print(f"Answer:\n{answer}\n")
        print(f"Keywords found  : {found}")
        print(f"Keywords missing: {missing}")
        score = len(found) / len(keywords) * 100
        print(f"Coverage score  : {score:.0f}%")
        print(f"Result          : {'PASS' if score >= 70 else 'FAIL'}\n")
        print("=" * 70 + "\n")

run_ground_truth_test(TEST_CASES)

GROUND TRUTH ACCURACY TEST

Test 1: What is the minimum penalty for insider trading under SEBI Act?
--------------------------------------------------
Answer:
The context provided does not mention the minimum penalty for insider trading under the SEBI Act [N]. It only mentions that a monetary penalty can be imposed for insider trading under Section 15G of the SEBI Act [2], but it does not specify the minimum amount [N].

Keywords found  : ['Section 15G', 'minimum penalty']
Keywords missing: ['ten lakh', '10,00,000']
Coverage score  : 50%
Result          : FAIL


Test 2: What are the grounds for specific performance of a contract?
--------------------------------------------------
Answer:
According to the context, the grounds for specific performance of a contract are as follows: 

The court may properly exercise discretion to decree specific performance in any case where the plaintiff has done substantial acts or suffered losses in consequence of a contract capable of specific performa

### QUick Manual Spot Check

In [ ]:
def spot_check(question: str, n_results: int = 6):
    """
    Print the raw retrieved chunks with full text so the audience
    can read the source themselves and verify the answer is grounded.
    """
    q_vec = model.encode([question], normalize_embeddings=True)[0].tolist()
    results = collection.query(
        query_embeddings=[q_vec], n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    print(f"\nRAW RETRIEVED SOURCES FOR: '{question}'\n")
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ), 1):
        print(f"SOURCE [{i}]  Similarity: {round(1-dist, 3)}")
        print(f"Title : {meta.get('title','?')}")
        print(f"Court : {meta.get('court','?')}  |  Year: {meta.get('year','?')}")
        print(f"Section: {meta.get('section','?')}")
        print(f"\nFULL TEXT:\n{doc}\n")
        print("-" * 70)

spot_check("What is the penalty for insider trading under SEBI regulations?")


RAW RETRIEVED SOURCES FOR: 'What is the penalty for insider trading under SEBI regulations?'

SOURCE [1]  Similarity: 0.802
Title : Rakesh Agrawal vs Securities Exchange Board Of India on 1 November, 2003
Court :   |  Year: 2003
Section: general

FULL TEXT:
114. Under the provisions of 
Section 11(2)(g)
 SEBI is to prevent insider trading and take such measures to protect the interest of investors as insider trading is per se a wrong and is prohibited. Insider Trading:-

----------------------------------------------------------------------
SOURCE [2]  Similarity: 0.801
Title : Rakesh Agrawal vs Securities Exchange Board Of India on 1 November, 2003
Court :   |  Year: 2003
Section: general

FULL TEXT:
118. It is clear from the bare reading of 
Regulation 3
 that the prohibition of insider trading by an insider is an absolute offence and that benefit or gain is not an ingredient of the offence. The SEBI (Insider Trading) Regulations, 1992

----------------------------------------------

In [14]:
import os

# Search the entire file system for InLegalBERT config
print("Searching for config.json...\n")

found = []
search_dirs = [
    os.path.expanduser("~/.cache"),
    "/root/.cache",
    "/content",
    "/tmp",
]

for search_dir in search_dirs:
    if os.path.exists(search_dir):
        for root, dirs, files in os.walk(search_dir):
            for file in files:
                if file == "config.json":
                    full_path = os.path.join(root, file)
                    found.append(full_path)
                    print(full_path)

if not found:
    print("No config.json found anywhere.")

Searching for config.json...

/root/.cache/huggingface/hub/models--law-ai--InLegalBERT/snapshots/b5ecfed8ed6cf9d25a3cb8225a8c52f161f7401a/config.json
/root/.cache/huggingface/hub/models--law-ai--InLegalBERT/snapshots/b5ecfed8ed6cf9d25a3cb8225a8c52f161f7401a/config.json
/content/drive/MyDrive/DSAI_project/checkpoints/Full-RAG-FSP-CoT/config.json
/content/drive/MyDrive/DSAI_project/checkpoints/FSP/config.json
/content/drive/MyDrive/DSAI_project/model_export/config.json


In [16]:
import json

# Replace this with the path printed above
config_path = "/root/.cache/huggingface/hub/models--law-ai--InLegalBERT/snapshots/b5ecfed8ed6cf9d25a3cb8225a8c52f161f7401a/config.json"

with open(config_path) as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

{
  "_name_or_path": "law-ai/InLegalBERT",
  "architectures": [
    "BertForPreTraining"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_ids": 0,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.17.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}
